In [1]:
## Clinical Conditions Summary
import sys
sys.path.append("..")
import pandas as pd
from src.cleaning import clean_patients, clean_encounters

patients = clean_patients(pd.read_csv("../data/raw/patients.csv"))

encounters = clean_encounters(pd.read_csv("../data/raw/encounters.csv"))

# a clinical state/diagnosis associated with the patient.
conditions = pd.read_csv("../data/raw/conditions.csv", dtype={"CODE": str})

conditions["START"] = pd.to_datetime(conditions["START"], errors="coerce")
conditions["STOP"]  = pd.to_datetime(conditions["STOP"],  errors="coerce")

conditions.shape
conditions.columns
conditions.info()
conditions.isna().sum()
conditions.head(10)



<class 'pandas.DataFrame'>
RangeIndex: 3784 entries, 0 to 3783
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   START        3784 non-null   datetime64[us]
 1   STOP         2818 non-null   datetime64[us]
 2   PATIENT      3784 non-null   str           
 3   ENCOUNTER    3784 non-null   str           
 4   SYSTEM       3784 non-null   str           
 5   CODE         3784 non-null   str           
 6   DESCRIPTION  3784 non-null   str           
dtypes: datetime64[us](2), str(5)
memory usage: 207.1 KB


,START,STOP,PATIENT,ENCOUNTER,SYSTEM,CODE,DESCRIPTION
0,1997-04-22,NaT,a1733070-046a-4506-bba6-47f32652e9d7,a1733070-046a-4506-13b1-f518d757cdd0,http://snomed.info/sct,224299000,Received higher education (finding)
1,1997-04-22,NaT,a1733070-046a-4506-bba6-47f32652e9d7,a1733070-046a-4506-13b1-f518d757cdd0,http://snomed.info/sct,713458007,Lack of access to transportation (finding)
2,1989-01-10,NaT,f9ba83f7-9940-16ae-0854-24bd34bf1843,f9ba83f7-9940-16ae-158c-5cd0e5a140b9,http://snomed.info/sct,160968000,Risk activity involvement (finding)
3,2013-05-14,NaT,a1733070-046a-4506-bba6-47f32652e9d7,a1733070-046a-4506-0d73-cf2822965f28,http://snomed.info/sct,266934004,Transport problem (finding)
4,2014-02-06,NaT,a1733070-046a-4506-bba6-47f32652e9d7,a1733070-046a-4506-175b-fc14062cdb07,http://snomed.info/sct,82423001,Chronic pain (finding)
5,2014-02-06,NaT,a1733070-046a-4506-bba6-47f32652e9d7,a1733070-046a-4506-175b-fc14062cdb07,http://snomed.info/sct,278860009,Chronic low back pain (finding)
6,2014-02-06,NaT,a1733070-046a-4506-bba6-47f32652e9d7,a1733070-046a-4506-175b-fc14062cdb07,http://snomed.info/sct,1121000119107,Chronic neck pain (finding)
7,1990-01-16,NaT,f9ba83f7-9940-16ae-0854-24bd34bf1843,f9ba83f7-9940-16ae-665a-6f37a1042b1a,http://snomed.info/sct,105531004,Housing unsatisfactory (finding)
8,1990-01-16,NaT,f9ba83f7-9940-16ae-0854-24bd34bf1843,f9ba83f7-9940-16ae-665a-6f37a1042b1a,http://snomed.info/sct,224299000,Received higher education (finding)
9,2003-02-04,NaT,f9ba83f7-9940-16ae-0854-24bd34bf1843,f9ba83f7-9940-16ae-66d7-aab90f06f82e,http://snomed.info/sct,87433001,Pulmonary emphysema (disorder)


In [2]:
# Patient-level vs record-level analysis

total_patients = patients['Id'].nunique()
patients_with_conditions = conditions['PATIENT'].nunique()

pct_patients_with_conditions = (patients_with_conditions / total_patients) * 100

print(f"Percentage of patients with at least one condition: {pct_patients_with_conditions:.2f}%")


Percentage of patients with at least one condition: 100.00%


In [3]:
# what are the most common conditions ?

condition_counts = conditions["DESCRIPTION"].value_counts().head(10)
condition_percentage = (condition_counts / len(conditions) * 100).round(2)

topconditions =  pd.DataFrame({
    'Count' : condition_counts,
    'Percentage' : condition_percentage
})
print(topconditions)


                                   Count  Percentage
DESCRIPTION                                         
Medication review due (situation)    784       20.72
Stress (finding)                     285        7.53
Full-time employment (finding)       275        7.27
Gingivitis (disorder)                270        7.14
Part-time employment (finding)       175        4.62
Social isolation (finding)           107        2.83
Limited social contact (finding)     106        2.80
Viral sinusitis (disorder)           103        2.72
Gingival disease (disorder)           85        2.25
Not in labor force (finding)          80        2.11


In [4]:
# categorized number of conditions 
conditions["category"] = (
    conditions["DESCRIPTION"].str.extract(r"\(([^)]+)\)$")[0].fillna("untagged")
)
conditions["category"].value_counts()



category
finding                    1709
disorder                   1202
situation                   842
morphologic abnormality      16
untagged                      8
person                        7
Name: count, dtype: int64

In [5]:
# conditions vs patients demographics
patients_with_conditions = conditions.groupby("PATIENT").size().sort_values(ascending=False)
patients_with_conditions.describe()

count    108.000000
mean      35.037037
std       38.892838
min        2.000000
25%       18.000000
50%       30.000000
75%       38.250000
max      319.000000
dtype: float64

In [6]:
# analysis  after merging with patients 
conditions_patients_merged = conditions.merge(
    patients[["Id", "GENDER", "age", "age_group", "BIRTHDATE", "CITY", "STATE"]], 
    left_on="PATIENT", right_on="Id", how="left").drop(columns=["Id"])

print(conditions_patients_merged.head(5))

# integrity check: 
assert len(conditions_patients_merged) == len(conditions)
print("unmatched patients:", conditions_patients_merged["GENDER"].isna().sum())


       START STOP                               PATIENT  \
0 1997-04-22  NaT  a1733070-046a-4506-bba6-47f32652e9d7   
1 1997-04-22  NaT  a1733070-046a-4506-bba6-47f32652e9d7   
2 1989-01-10  NaT  f9ba83f7-9940-16ae-0854-24bd34bf1843   
3 2013-05-14  NaT  a1733070-046a-4506-bba6-47f32652e9d7   
4 2014-02-06  NaT  a1733070-046a-4506-bba6-47f32652e9d7   

                              ENCOUNTER                  SYSTEM       CODE  \
0  a1733070-046a-4506-13b1-f518d757cdd0  http://snomed.info/sct  224299000   
1  a1733070-046a-4506-13b1-f518d757cdd0  http://snomed.info/sct  713458007   
2  f9ba83f7-9940-16ae-158c-5cd0e5a140b9  http://snomed.info/sct  160968000   
3  a1733070-046a-4506-0d73-cf2822965f28  http://snomed.info/sct  266934004   
4  a1733070-046a-4506-175b-fc14062cdb07  http://snomed.info/sct   82423001   

                                  DESCRIPTION category GENDER   age  \
0         Received higher education (finding)  finding      M  47.5   
1  Lack of access to transportatio

In [7]:

# How many conditions does each patient have?
conditions_per_patient = (
    conditions_patients_merged.groupby("PATIENT")
    .size()
    .sort_values(ascending=False)
)
print(conditions_per_patient.head(10))
print(conditions_patients_merged.describe())
print((conditions_per_patient == 1).sum())
print((conditions_per_patient > 5).sum())
print(conditions_per_patient.idxmax())
print(conditions_per_patient.max())
patients_id = conditions_per_patient.idxmax()
conditions[conditions["PATIENT"] == patients_id]
print(f"Patient with ID {patients_id} has the most conditions: {conditions_per_patient.max()} conditions.")

PATIENT
5d84e6a3-b4bd-63d6-57c5-cada0916490d    319
7ad140ab-bfae-c3ae-20a3-2244b1c4d0e2    219
688c8453-ba6f-7dec-03c3-eaa27d6df1a4    128
01a006ce-6457-50c2-8e0a-fb58fc310a86    109
8d7f6a31-31ba-da9c-2b57-03ee0f7577a0    103
c1e9a9fb-45ec-4ee4-c946-4ffc5dfa93ea     93
53e4891a-9108-67ef-d973-3b6a98404249     78
ba234ff2-cefe-dfee-935a-8ea2378da8c2     57
885c1eeb-1b4c-8a3a-bfc7-a68897e470a6     55
2eb14889-46f2-06de-4f6e-fea5634e0d85     51
dtype: int64
                            START                        STOP          age  \
count                        3784                        2818  3784.000000   
mean   2017-07-16 13:29:48.583509  2020-11-24 20:28:26.742370    49.948441   
min           1952-10-22 00:00:00         2004-12-31 00:00:00     1.000000   
25%           2016-05-05 12:00:00         2018-06-18 00:00:00    38.000000   
50%           2020-02-01 00:00:00         2021-01-11 12:00:00    50.800000   
75%           2023-02-02 00:00:00         2023-09-25 00:00:00    67.600

In [8]:
#  Which conditions drive the most demand?
disorders = conditions_patients_merged[
    conditions_patients_merged["category"].isin(["disorder", "untagged"])
].copy()
total_patients = patients['Id'].nunique()

prevalence = (
    disorders.groupby("DESCRIPTION")["PATIENT"].nunique()
    .div(total_patients).mul(100).round(1)
    .sort_values(ascending=False)
    .astype(str) + "%"
)
prevalence.head(10)

DESCRIPTION
Gingivitis (disorder)                 75.9%
Viral sinusitis (disorder)            61.1%
Acute viral pharyngitis (disorder)    46.3%
Primary dental caries (disorder)      43.5%
Gingival disease (disorder)           43.5%
Anemia (disorder)                     38.0%
Acute bronchitis (disorder)           38.0%
Chronic sinusitis (disorder)          25.0%
Fracture of bone (disorder)           18.5%
Essential hypertension (disorder)     18.5%
Name: PATIENT, dtype: str

In [9]:
# Does complexity rise with age?

disorders_per_patient_by_age = (
    disorders.groupby(["age_group", "PATIENT"], observed=True)["CODE"].nunique()
    .groupby("age_group", observed=True).mean().round(1)
)
disorders_per_patient_by_age


age_group
young         4.3
adult         7.2
middleage     9.0
elder        13.0
Name: CODE, dtype: float64

In [10]:
# Do social factors predict disease? 
SOCIAL_RISK = [
    "Social isolation", "Limited social contact", "Not in labor force",
    "Victim of intimate partner abuse", "Reports of violence in the environment",
]
burden = disorders.groupby("PATIENT")["CODE"].nunique().sort_values(ascending=False)

at_risk = set(conditions_patients_merged.loc[
    conditions_patients_merged["DESCRIPTION"].str.contains("|".join(SOCIAL_RISK), case=False),
    "PATIENT"
])

print("with social risk :", round(burden[burden.index.isin(at_risk)].mean(), 1), f"(n={len(at_risk)})")
print("without :", round(burden[~burden.index.isin(at_risk)].mean(), 1))


with social risk : 9.5 (n=76)
without : 5.2


In [11]:
# At what age do conditions first appear?

disorders["age_at_diagnosis"] = (
    (disorders["START"] - disorders["BIRTHDATE"]).dt.days / 365.25
).round(1)

first_dx = (
    disorders.groupby(["PATIENT", "DESCRIPTION"])["age_at_diagnosis"]
    .min().reset_index()
)

(first_dx[first_dx["DESCRIPTION"].isin(prevalence.head(5).index)]
    .groupby("DESCRIPTION")["age_at_diagnosis"]
    .agg(["mean", "min", "max"]).round(1))


,mean,min,max
DESCRIPTION,,,
Acute viral pharyngitis (disorder),33.8,0.9,73.3
Gingival disease (disorder),41.5,4.9,78.5
Gingivitis (disorder),37.9,8.0,78.4
Primary dental caries (disorder),40.5,3.9,78.5
Viral sinusitis (disorder),33.1,0.4,74.8


In [12]:
disorders.groupby(["GENDER", "PATIENT"])["CODE"].nunique().groupby("GENDER").mean().round(1)


GENDER
F    8.3
M    8.4
Name: CODE, dtype: float64